In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [5]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_markdown.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 4.66 s
Wall time: 4.72 s


180

In [6]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준 | 세율 |\n\n|---|---|\n\n| 1,400만원 이하 | 과세표준의 6퍼센트 |\n\n| 1,400만원 초과 5,000만원 이하 | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트) |\n\n| 5,000만원 초과 8,800만원 이하 | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하 | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트) |\n\n| 1억5천만원 초과 3억원 이하 | 3,706만원 + (1

In [7]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)

In [8]:
len(embedding.embed_query("소득세법"))

4096

In [9]:
from pinecone import Pinecone
pc = Pinecone()
[index.name for index in pc.list_indexes()]

['tax-index', 'tax-index-table', 'tax-index-openai']

In [10]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# 0. pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-markdown'
dimension = len(embedding.embed_query("차원 확인용"))

# 2. 인덱스 생성(해당 인덱스가 없을 경우)
existing_indexes = [index.name for index in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name, 
        dimension=dimension,
        metric='cosine',  # 유사도 측정 방식
        spec=ServerlessSpec(
            cloud='aws',    # 무료 플랜은 aws
            region='us-east-1' # 무료플랜은 us-east-1만 가능
        )        
    )
    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초 대기
    print(f'인덱스 {index_name}를 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index-markdown를 생성하였습니다


In [11]:
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index', 'tax-index-table', 'tax-index-openai', 'tax-index-markdown']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [12]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한 번
from pinecone import Pinecone
pc = Pinecone()
from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할 때
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 156 ms
Wall time: 1.19 s


# 2. 답변 생성 전 retrieval 확인

In [13]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=2) # 기본k값은 4

In [14]:
len(retrieved_docs)

2

In [15]:
retrieved_docs[1]

Document(id='db943574-db63-4d3f-828a-43dee89c61ec', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제58조(재해손실세액공제) ① 사업자가 해당 과세기간에 천재지변이나 그 밖의 재해(이하 “재해”라 한다)로 대통령령으로 정하는 자산총액(이하 이 항에서 “자산총액”이라 한다)의 100분의 20 이상에 해당하는 자산을 상실하여 납세가 곤란하다고 인정되는 경우에는 다음 각 호의 소득세액(사업소득에 대한 소득세액을 말한다. 이하 이 조에서 같다)에 그 상실된 가액이 상실 전의 자산총액에서 차지하는 비율(이하 이 조에서 “자산상실비율”이라 한다)을 곱하여 계산한 금액(상실된 자산의 가액을 한도로 한다)을 그 세액에서 공제한다. 이 경우 자산의 가액에는 토지의 가액을 포함하지 아니한다. <개정 2020. 12. 29.>\n\n1. 재해 발생일 현재 부과되지 아니한 소득세와 부과된 소득세로서 미납된 소득세액\n\n2. 재해 발생일이 속하는 과세기간의 소득에 대한 소득세액\n\n② 제1항의 경우에 제56조, 제56조의2, 제57조 및 제57조의2에 따라 공제할 세액이 있을 때에는 이를 공제한 후의 세액을 소득세액으로 하여 제1항을 적용한다.<개정 2022. 12. 31.>\n\n③ 제1항에 따른 공제를 “재해손실세액공제”라 한다.\n\n④ 재해손실세액공제를 적용받으려는 자는 대통령령으로 정하는 바에 따라 관할 세무서장에게 신청할 수 있다.\n\n⑤ 관할 세무서장이 제4항의 신청을 받았을 때에는 그 공제할 세액을 결정하여 신청인에게 알려야 한다.\n\n⑥ 제4항의 신청이 없는 경우에도 제1항을 적용한다.\n\n⑦ 집단적으로 재해가 발생한 경우에는 대통령령으로 정하는 바에 따라 관할 세무서장이 조사결정한 자산상실비율에 따라 제1항을 적용한다.\n\n⑧ 재해손실세액공제에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\

In [16]:
retriever = database.as_retriever( 
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")

In [17]:
retrieved_docs[1]

Document(id='db943574-db63-4d3f-828a-43dee89c61ec', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제58조(재해손실세액공제) ① 사업자가 해당 과세기간에 천재지변이나 그 밖의 재해(이하 “재해”라 한다)로 대통령령으로 정하는 자산총액(이하 이 항에서 “자산총액”이라 한다)의 100분의 20 이상에 해당하는 자산을 상실하여 납세가 곤란하다고 인정되는 경우에는 다음 각 호의 소득세액(사업소득에 대한 소득세액을 말한다. 이하 이 조에서 같다)에 그 상실된 가액이 상실 전의 자산총액에서 차지하는 비율(이하 이 조에서 “자산상실비율”이라 한다)을 곱하여 계산한 금액(상실된 자산의 가액을 한도로 한다)을 그 세액에서 공제한다. 이 경우 자산의 가액에는 토지의 가액을 포함하지 아니한다. <개정 2020. 12. 29.>\n\n1. 재해 발생일 현재 부과되지 아니한 소득세와 부과된 소득세로서 미납된 소득세액\n\n2. 재해 발생일이 속하는 과세기간의 소득에 대한 소득세액\n\n② 제1항의 경우에 제56조, 제56조의2, 제57조 및 제57조의2에 따라 공제할 세액이 있을 때에는 이를 공제한 후의 세액을 소득세액으로 하여 제1항을 적용한다.<개정 2022. 12. 31.>\n\n③ 제1항에 따른 공제를 “재해손실세액공제”라 한다.\n\n④ 재해손실세액공제를 적용받으려는 자는 대통령령으로 정하는 바에 따라 관할 세무서장에게 신청할 수 있다.\n\n⑤ 관할 세무서장이 제4항의 신청을 받았을 때에는 그 공제할 세액을 결정하여 신청인에게 알려야 한다.\n\n⑥ 제4항의 신청이 없는 경우에도 제1항을 적용한다.\n\n⑦ 집단적으로 재해가 발생한 경우에는 대통령령으로 정하는 바에 따라 관할 세무서장이 조사결정한 자산상실비율에 따라 제1항을 적용한다.\n\n⑧ 재해손실세액공제에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\

In [18]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
# print(retrieved_doc)

In [19]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [20]:
from openai import OpenAI
client = OpenAI()
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""- [context]를 참고해서 사용자의 질문에 답변해 주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [21]:
result = response.choices[0].message.content
print(result)

연봉이 5천만원인 직장인의 소득세를 계산하기 위해서는 먼저 종합소득 과세표준에 따른 세율을 적용해야 합니다.

1. **과세표준 계산**: 연봉 5천만원은 종합소득 과세표준으로 그대로 적용됩니다.

2. **세율 적용**:
   - 1,400만원 이하: 1,400만원 × 6% = 84만원
   - 1,400만원 초과 5,000만원 이하: (5,000만원 - 1,400만원) × 15% = 3,600만원 × 15% = 540만원

3. **종합소득산출세액 계산**:
   - 총 세액 = 84만원 + 540만원 = 624만원

따라서, 연봉이 5천만원인 직장인의 소득세는 **624만원**입니다.


In [22]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-mini')

In [23]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변 (low, medium, high)
)

In [24]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고하여 사용자의 질문에 답변해 주세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [26]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세는 **근로소득공제, 인적공제, 기타 공제항목, 세액공제** 등을 거쳐 산정되므로, 정확한 금액은 개인별 공제 상황에 따라 달라집니다. 다만, 질문하신 내용과 주신 법조문(제55조 세율, 제59조 근로소득세액공제)을 바탕으로 **다른 공제가 없다고 가정할 때의 기준 계산 흐름**을 단계별로 설명해 드리겠습니다.

---

### 1. 근로소득금액 계산 (근로소득공제)
직장인의 `총급여(연봉)`에서 법정 근로소득공제를 먼저 차감합니다. (현행 기준)
- 총급여 4,500만원 초과 ~ 1억원 이하 구간: `1,200만원 + (총급여 - 4,500만원) × 40%`
- 5,000만원 적용 시: `1,200만원 + (500만원 × 0.4) = 1,400만원`
- **근로소득금액** = 5,000만원 - 1,400만원 = **3,600만원**

### 2. 종합소득 과세표준 산정
과세 대상이 되는 금액을 구하기 위해 근로소득금액에서 기본공제(인적공제) 등을 차감합니다.
- 본인 기본공제: 1인당 150만원 (배우자·부양가족이 없을 경우 본인만 적용)
- **과세표준** = 3,600만원 - 150만원 = **3,450만원**
※ 실제는 보험료, 의료비, 교육비, 신용카드 사용액, 주택자금, 개인연금 등 `특별공제` 또는 `기타 공제`를 추가로 차감할 수 있으며, 이에 따라 과세표준은 더 낮아집니다.

### 3. 종합소득 산출세액 계산 (제55조 세율 적용)
주신 제55조 제1항의 세율표를 과세표준 3,450만원에 적용합니다.
- 구간: 1,400만원 초과 5,000만원 이하
- 세율 구조: `84만원 + (1,400만원 초과금액의 15%)`
- 계산: `84만원 + (3,450만원 - 1,400만원) × 15%`  
  = `84만원 + (2,050만원 × 0.15)`  
  = `84만원 + 307.5만원`  
  = **387.5만원** (종합소득산출세액)

### 4. 근로소득세액공제 적용 (제59조)
주어진 제59조 제2항 

## Langchain

In [1]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
# pc = Pinecone()
index_name = 'tax-index-markdown'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [2]:
# 6. RAG 렝체인 구성 (LCEL방식)
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

2026년 기준 연봉 5,000만 원 직장인의 소득세는 부양가족 수, 사용 중인 공제 항목(신용카드·의료비·교육비 등), 기타 소득에 따라 매년 크게 달라지므로 단일 금액을 특정할 수 없습니다. 국세청 '연말정산 간소화 서비스'나 '소득세 계산기'에 실제 부양가족과 공제·지출 내역을 입력하면 정확한 세액을 바로 산출할 수 있습니다. 따라서 정확한 금액을 알기 위해서는 본인의 구체적인 연말정산 내역을 기준으로 계산해 보셔야 합니다.


# 4. 키워드 사전 활용

In [3]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
# query = '종합소득이 5,000만원인 거주자의 소득세는 얼마인가요?'

In [4]:
import numpy as np
def cosine_similarity(vec1, vec2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)
vec1 = embedding.embed_query("직장인")
vec2 = embedding.embed_query("거주자")
cosine_similarity(vec1, vec2)

np.float64(0.5402861885922909)

In [5]:
vec1 = embedding.embed_query("연봉")
vec2 = embedding.embed_query("종합소득")
cosine_similarity(vec1, vec2)

np.float64(0.566316344244509)

In [6]:
vec1 = embedding.embed_query("5천")
vec2 = embedding.embed_query("5,000")
cosine_similarity(vec1, vec2)

np.float64(0.8728291248383045)

In [7]:
# 키워드 사전 : 일반사용자 단어 -> 소득세법 용어
from langchain_core.prompts import ChatPromptTemplate
dictionary = ["연봉->종합소득", "사람을 나타내는 표현->거주자", "5천->5,000"]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요
- 사전 : {dictionary}
- 질문 : {{question}}""")

In [8]:
strOutputParser = StrOutputParser()
strOutputParser.invoke(llm.invoke(prompt.invoke({"question":"소득이 높은 남자가 있습니다"})))

'종합소득이 높은 거주자가 있습니다'

In [9]:
# 위를 langchain으로
dictionary_chain = prompt | llm | StrOutputParser()
#dictionary_chain.invoke({"question":"연봉이 5천만원인 직장인은?"})
dictionary_chain.invoke("연봉이 5천만원인 직장인은?")

'종합소득이 5,000만 원인 거주자는?'

In [10]:
new_chain = dictionary_chain | rag_chain

In [11]:
new_chain.invoke("연봉이 5천만원인 직장인의 소득세는 얼마예요?")
# new_chain.invoke({"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"})

'질문하신 내용만으로는 정확한 소득세를 계산할 수 없습니다. 소득세 금액은 소득공제 적용 여부, 누진세율 구간, 세액공제 항목 등에 따라 달라지며, 제공된 문맥이 없어 구체적인 산출 근거를 확인할 수 없기 때문입니다. 정확한 세액을 알고 싶으시면 종합소득세 신고서 작성 또는 국세청 세액계산기를 이용하시기 바랍니다.'

In [ ]:
# 입력 {"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"}
#       ↓
# dictionary_chain에 의해 질문 수정(documents용어 수정)
#       ↓
#  "개선된 질문"
#       ↓
#  retriever
#       ↓ 
# format_documents
#       ↓ 
# rag_chain의 prompt + llm
#       ↓ 
# strOutputParser()